# Eksperyment: EWMA / recency weighting (Sprint 3c)

## Cel
Zamiast prostej sredniej z 10 ostatnich meczow (SMA) uzyc **wykladniczego wazenia** -- starsze mecze
maleja gladko (alpha=0.18, half-life ~3.5 meczu). Nadpisujemy cechy formy i serwisu (te same nazwy),
nie dodajemy nowych. To zmiana REPREZENTACJI cech.

In [1]:
import sys, io, contextlib, runpy
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
sys.path.insert(0, str(Path("../src").resolve()))
from tennis_model_ewma_ablation import compute_ewma_features, OVERWRITE_COLS, ALPHA, data_file, TARGET_YEAR, HISTORY_START_YEAR
print("alpha:", ALPHA, "| nadpisywane kolumny:", len(OVERWRITE_COLS))

alpha: 0.18 | nadpisywane kolumny: 20


## 1. Reuse baseline pipeline

In [2]:
BASE = Path("../src/tennis_model.py").resolve()
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    ns = runpy.run_path(str(BASE))
symmetrize_data = ns["symmetrize_data"]
compute_symmetric_match_evaluation = ns["compute_symmetric_match_evaluation"]
evaluate_calibration_quality = ns["evaluate_calibration_quality"]
baseline_search = ns["search"]
RANDOM_STATE = ns["RANDOM_STATE"]
base_features = list(ns["features"]); cols_base = list(ns["cols_base"])
df_train_raw = ns["df_train_raw"].copy(); df_val_raw = ns["df_val_raw"].copy(); df_test_raw = ns["df_test_raw"].copy()
baseline_val_acc = float(ns["val_acc"]); baseline_test_acc = float(ns["test_acc"]); baseline_match_acc = float(ns["match_accuracy"])
print(f"Baseline: val={baseline_val_acc:.4f}  test={baseline_test_acc:.4f}  match={baseline_match_acc:.4f}  (cech: {len(base_features)})")
features = base_features

Baseline: val=0.6106  test=0.6604  match=0.6566  (cech: 40)


## 2. Liczenie cech EWMA z chronologii (leakage-safe)
`compute_ewma_features` przetwarza historie + rok docelowy incrementalnie, utrzymujac stan EWMA
per gracz, i zapisuje pre-match wartosci (forma/serwis/surface_form).

In [3]:
full_target = pd.read_csv(data_file(TARGET_YEAR))
full_target["tourney_date"] = pd.to_datetime(full_target["tourney_date"], format="%Y%m%d")
full_target = full_target.sort_values(["tourney_date","match_num"]).reset_index(drop=True)
full_target_base = full_target[cols_base].dropna(subset=cols_base).reset_index(drop=True)
n_train,n_val,n_test = len(df_train_raw),len(df_val_raw),len(df_test_raw)
assert len(full_target_base)==n_train+n_val+n_test
ewma = compute_ewma_features(full_target_base, cols_base)
e_train=ewma.iloc[:n_train].reset_index(drop=True); e_val=ewma.iloc[n_train:n_train+n_val].reset_index(drop=True); e_test=ewma.iloc[n_train+n_val:].reset_index(drop=True)
print("Kolumny EWMA:", OVERWRITE_COLS[:4], "...")

Kolumny EWMA: ['w_form', 'l_form', 'w_surface_form', 'l_surface_form'] ...


## 3. Nadpisanie cech SMA -> EWMA, symetryzacja, retrening

In [4]:
def overwrite(df_raw, e):
    df_raw = df_raw.copy().reset_index(drop=True)
    for col in OVERWRITE_COLS: df_raw[col] = e[col].to_numpy()
    return df_raw
df_train_raw=overwrite(df_train_raw,e_train); df_val_raw=overwrite(df_val_raw,e_val); df_test_raw=overwrite(df_test_raw,e_test)
train_data=symmetrize_data(df_train_raw,shuffle=True); val_data=symmetrize_data(df_val_raw,shuffle=True); test_data=symmetrize_data(df_test_raw,shuffle=True)
X_train,y_train=train_data[features],train_data["y"]; X_val,y_val=val_data[features],val_data["y"]; X_test,y_test=test_data[features],test_data["y"]
best_rf = RandomForestClassifier(**baseline_search.best_params_, n_jobs=-1, random_state=RANDOM_STATE).fit(X_train,y_train)
val_acc=float(accuracy_score(y_val,best_rf.predict(X_val))); test_acc=float(accuracy_score(y_test,best_rf.predict(X_test)))
proba=best_rf.predict_proba(X_test)[:,1]; test_data["p1_win_probability"]=proba
_,match_acc=compute_symmetric_match_evaluation(test_data); q=evaluate_calibration_quality(y_test.to_numpy(),proba)
print(f"baseline (SMA)  val={baseline_val_acc:.4f}  test={baseline_test_acc:.4f}  match={baseline_match_acc:.4f}")
print(f"EWMA            val={val_acc:.4f}  test={test_acc:.4f}  match={match_acc:.4f}  Brier={q['brier_score']:.4f}")
print(f"DELTA match: {match_acc-baseline_match_acc:+.4f}")

baseline (SMA)  val=0.6106  test=0.6604  match=0.6566
EWMA            val=0.6361  test=0.6547  match=0.6491  Brier=0.2163
DELTA match: -0.0075


## Wnioski
EWMA daje **niespojny** wynik: val potrafi wzrosnac, ale test/match prawie sie nie zmieniaja. Powod:
po Sprincie 1 baseline ma juz twarde okno 365 dni, ktore wychwytuje wiekszosc zysku z recency.
Wniosek: EWMA nie poprawia modelu w sposob istotny.